# ⚖️ Notebook 06: Fairness Audit & Bias Detection
## Pigmentation-Stratified Performance Analysis

**Objectives:**
1. ✅ Estimate pigmentation proxy from retinal images
2. ✅ Compute per-group performance metrics
3. ✅ Calculate fairness metrics (Equalized Odds, Demographic Parity)
4. ✅ Generate comprehensive fairness report

**Why Fairness Audit?**
- Medical AI must perform equitably across demographics
- Retinal pigmentation varies with ethnicity
- Bias in training data → Bias in predictions
- Regulatory requirement for medical devices

**Expected Output:**
- `artifacts/fairness_report.png`
- `results/fairness_audit.json`
- `src/fairness/audit.py`

---

## 1. Setup & Imports

In [ ]:
# Core imports
import os
import sys
import json
from pathlib import Path

# Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F

# Image processing
import cv2
import numpy as np
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Data & Metrics
import pandas as pd
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, 
    confusion_matrix, classification_report
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Progress bar
from tqdm.notebook import tqdm

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"✅ Using device: {device}")

In [ ]:
# Define paths
PROJECT_ROOT = Path("/Users/shivasaivemula/ALP Project/deep-retina-grade")
DATA_ROOT = PROJECT_ROOT / "aptos2019-blindness-detection"

TRAIN_IMAGES_DIR = DATA_ROOT / "train_images"
SPLITS_DIR = PROJECT_ROOT / "splits"
MODELS_DIR = PROJECT_ROOT / "models"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RESULTS_DIR = PROJECT_ROOT / "results"
SRC_DIR = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_DIR))

# Load test split
df_test = pd.read_csv(SPLITS_DIR / "test.csv")
print(f"📊 Test samples: {len(df_test):,}")

In [ ]:
# Load model and preprocessor
from models.efficientnet import RetinaModel
from preprocessing.preprocess import RetinaPreprocessor

# Load model - Using EfficientNet-B0 (optimized for M1 Mac)
model = RetinaModel(num_classes=5, pretrained=False, backbone='efficientnet_b0')
checkpoint = torch.load(MODELS_DIR / 'efficientnet_b0_best.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

# Initialize preprocessor
preprocessor = RetinaPreprocessor(img_size=224)

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

transform = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

print(f"✅ Model loaded (EfficientNet-B0)!")
print(f"   Best Kappa: {checkpoint['best_kappa']:.4f}")

## 2. Pigmentation Proxy Estimation

Since APTOS dataset lacks demographic labels, we use **fundus luminance** as a proxy for retinal pigmentation:
- Lower mean brightness → Darker pigmentation
- Higher mean brightness → Lighter pigmentation

This is an imperfect proxy but commonly used in fairness research.

In [ ]:
def estimate_pigmentation(img_path, preprocessor=None):
    """
    Estimate retinal pigmentation from fundus image.
    
    Uses mean luminance in LAB color space as proxy.
    Higher L → Lighter pigmentation; Lower L → Darker pigmentation
    
    Args:
        img_path: Path to fundus image
        preprocessor: Optional preprocessor
        
    Returns:
        luminance: Mean luminance value [0, 100]
    """
    # Read image
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Crop to retina region (center 80%)
    h, w = img.shape[:2]
    margin = int(min(h, w) * 0.1)
    img_cropped = img[margin:h-margin, margin:w-margin]
    
    # Convert to LAB
    lab = cv2.cvtColor(img_cropped, cv2.COLOR_RGB2LAB)
    
    # Extract L channel
    L = lab[:, :, 0]
    
    # Mask out black borders
    mask = L > 10
    
    if mask.sum() == 0:
        return np.nan
    
    # Mean luminance of retina region
    mean_luminance = L[mask].mean()
    
    return float(mean_luminance)

# Test on a few images
print("Testing pigmentation estimation...")
for idx in [0, 100, 200]:
    img_path = TRAIN_IMAGES_DIR / f"{df_test.iloc[idx]['id_code']}.png"
    lum = estimate_pigmentation(img_path)
    print(f"  {df_test.iloc[idx]['id_code']}: L = {lum:.1f}")

In [ ]:
# Compute pigmentation for all test images
print("\nEstimating pigmentation for test set...")
pigmentation_values = []

for idx, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Computing luminance"):
    img_path = TRAIN_IMAGES_DIR / f"{row['id_code']}.png"
    lum = estimate_pigmentation(img_path)
    pigmentation_values.append(lum)

df_test['luminance'] = pigmentation_values
print(f"\n✅ Computed luminance for {len(df_test)} images")
print(f"   Mean: {df_test['luminance'].mean():.1f}")
print(f"   Std: {df_test['luminance'].std():.1f}")

In [ ]:
# Stratify into pigmentation groups
def stratify_pigmentation(luminance, bins=3):
    """
    Stratify images by pigmentation level.
    
    Args:
        luminance: Luminance value
        bins: Number of bins (default: 3 for Light/Medium/Dark)
        
    Returns:
        group: 'Light', 'Medium', or 'Dark'
    """
    if pd.isna(luminance):
        return 'Unknown'
    
    # Using percentiles for equal-sized groups
    q33 = df_test['luminance'].quantile(0.33)
    q66 = df_test['luminance'].quantile(0.66)
    
    if luminance < q33:
        return 'Dark'
    elif luminance < q66:
        return 'Medium'
    else:
        return 'Light'

df_test['pigmentation_group'] = df_test['luminance'].apply(stratify_pigmentation)

print("\n📊 Pigmentation Distribution:")
print(df_test['pigmentation_group'].value_counts())

In [ ]:
# Visualize pigmentation distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of luminance
axes[0].hist(df_test['luminance'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(df_test['luminance'].quantile(0.33), color='r', linestyle='--', label='33rd percentile')
axes[0].axvline(df_test['luminance'].quantile(0.66), color='g', linestyle='--', label='66th percentile')
axes[0].set_xlabel('Mean Luminance (L)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Distribution of Retinal Luminance', fontsize=14, fontweight='bold')
axes[0].legend()

# Sample images from each group
axes[1].axis('off')
axes[1].set_title('Example Images by Pigmentation Group', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'pigmentation_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: {ARTIFACTS_DIR / 'pigmentation_distribution.png'}")

## 3. Get Predictions for All Test Images

In [ ]:
@torch.no_grad()
def predict_batch(model, image_paths, preprocessor, transform, device):
    """
    Get predictions for a batch of images.
    
    Returns:
        predictions: List of predicted classes
        confidences: List of prediction confidences
    """
    model.eval()
    predictions = []
    confidences = []
    
    for img_path in tqdm(image_paths, desc="Predicting"):
        # Preprocess
        img = preprocessor.preprocess(img_path)
        img_uint8 = (img * 255).astype(np.uint8)
        
        # Transform
        transformed = transform(image=img_uint8)
        img_tensor = transformed['image'].unsqueeze(0).to(device)
        
        # Predict
        output = model(img_tensor)
        probs = F.softmax(output, dim=1)
        pred = output.argmax(dim=1).item()
        conf = probs[0, pred].item()
        
        predictions.append(pred)
        confidences.append(conf)
    
    return predictions, confidences

# Get predictions
image_paths = [TRAIN_IMAGES_DIR / f"{row['id_code']}.png" for _, row in df_test.iterrows()]
predictions, confidences = predict_batch(model, image_paths, preprocessor, transform, device)

df_test['prediction'] = predictions
df_test['confidence'] = confidences

# Overall accuracy
overall_acc = accuracy_score(df_test['diagnosis'], df_test['prediction'])
overall_qwk = cohen_kappa_score(df_test['diagnosis'], df_test['prediction'], weights='quadratic')

print(f"\n📊 Overall Performance:")
print(f"   Accuracy: {overall_acc:.2%}")
print(f"   QWK: {overall_qwk:.3f}")

## 4. Per-Group Performance Analysis

In [ ]:
def compute_group_metrics(df, group_col, pred_col='prediction', true_col='diagnosis'):
    """
    Compute metrics for each group.
    
    Returns:
        metrics_df: DataFrame with per-group metrics
    """
    groups = df[group_col].unique()
    results = []
    
    for group in groups:
        if group == 'Unknown':
            continue
            
        mask = df[group_col] == group
        y_true = df.loc[mask, true_col]
        y_pred = df.loc[mask, pred_col]
        
        acc = accuracy_score(y_true, y_pred)
        qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
        
        # Sensitivity (true positive rate) for DR detection (grade >= 2)
        y_true_binary = (y_true >= 2).astype(int)
        y_pred_binary = (y_pred >= 2).astype(int)
        
        tp = ((y_true_binary == 1) & (y_pred_binary == 1)).sum()
        fn = ((y_true_binary == 1) & (y_pred_binary == 0)).sum()
        fp = ((y_true_binary == 0) & (y_pred_binary == 1)).sum()
        tn = ((y_true_binary == 0) & (y_pred_binary == 0)).sum()
        
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        results.append({
            'Group': group,
            'N': mask.sum(),
            'Accuracy': acc,
            'QWK': qwk,
            'Sensitivity': sensitivity,
            'Specificity': specificity
        })
    
    return pd.DataFrame(results)

# Compute per-group metrics
group_metrics = compute_group_metrics(df_test, 'pigmentation_group')
print("\n📊 Per-Group Performance (by Pigmentation):")
print("="*70)
print(group_metrics.to_string(index=False))

## 5. Fairness Metrics Calculation

In [ ]:
def compute_fairness_metrics(group_metrics_df):
    """
    Compute fairness metrics from group performance.
    
    Metrics:
    - Demographic Parity Ratio: min(accuracy)/max(accuracy)
    - Equalized Odds Ratio: min(sensitivity)/max(sensitivity)
    - Max Disparity: max difference in any metric
    
    Returns:
        fairness_metrics: Dictionary of fairness metrics
    """
    metrics = {}
    
    # Demographic Parity (accuracy parity)
    acc_min = group_metrics_df['Accuracy'].min()
    acc_max = group_metrics_df['Accuracy'].max()
    metrics['demographic_parity_ratio'] = acc_min / acc_max if acc_max > 0 else 0
    metrics['accuracy_disparity'] = acc_max - acc_min
    
    # Equalized Odds (sensitivity/specificity parity)
    sens_min = group_metrics_df['Sensitivity'].min()
    sens_max = group_metrics_df['Sensitivity'].max()
    metrics['equalized_odds_ratio'] = sens_min / sens_max if sens_max > 0 else 0
    metrics['sensitivity_disparity'] = sens_max - sens_min
    
    # Specificity disparity
    spec_min = group_metrics_df['Specificity'].min()
    spec_max = group_metrics_df['Specificity'].max()
    metrics['specificity_disparity'] = spec_max - spec_min
    
    # QWK disparity
    qwk_min = group_metrics_df['QWK'].min()
    qwk_max = group_metrics_df['QWK'].max()
    metrics['qwk_disparity'] = qwk_max - qwk_min
    
    # Worst group
    metrics['worst_group_accuracy'] = group_metrics_df.loc[group_metrics_df['Accuracy'].idxmin(), 'Group']
    metrics['worst_group_sensitivity'] = group_metrics_df.loc[group_metrics_df['Sensitivity'].idxmin(), 'Group']
    
    # Fairness thresholds (commonly used)
    metrics['passes_80_rule_accuracy'] = metrics['demographic_parity_ratio'] >= 0.8
    metrics['passes_80_rule_sensitivity'] = metrics['equalized_odds_ratio'] >= 0.8
    
    return metrics

fairness_metrics = compute_fairness_metrics(group_metrics)

print("\n⚖️ Fairness Metrics:")
print("="*60)
print(f"\nDemographic Parity (Accuracy):")
print(f"   Ratio: {fairness_metrics['demographic_parity_ratio']:.3f}")
print(f"   Disparity: {fairness_metrics['accuracy_disparity']:.2%}")
print(f"   Passes 80% Rule: {'✓' if fairness_metrics['passes_80_rule_accuracy'] else '✗'}")

print(f"\nEqualized Odds (Sensitivity):")
print(f"   Ratio: {fairness_metrics['equalized_odds_ratio']:.3f}")
print(f"   Disparity: {fairness_metrics['sensitivity_disparity']:.2%}")
print(f"   Passes 80% Rule: {'✓' if fairness_metrics['passes_80_rule_sensitivity'] else '✗'}")

print(f"\nWorst Performing Groups:")
print(f"   By Accuracy: {fairness_metrics['worst_group_accuracy']}")
print(f"   By Sensitivity: {fairness_metrics['worst_group_sensitivity']}")

## 6. Visualize Fairness Report

In [ ]:
# Create comprehensive fairness report visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Color palette
colors = {'Light': '#FFD700', 'Medium': '#FF8C00', 'Dark': '#8B4513'}

# --- Plot 1: Accuracy by Group ---
ax1 = axes[0, 0]
groups = group_metrics['Group'].tolist()
accuracies = group_metrics['Accuracy'].tolist()
bar_colors = [colors.get(g, '#808080') for g in groups]

bars = ax1.bar(groups, accuracies, color=bar_colors, edgecolor='black', linewidth=2)
ax1.axhline(y=overall_acc, color='red', linestyle='--', linewidth=2, label=f'Overall ({overall_acc:.2%})')
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Accuracy by Pigmentation Group', fontsize=14, fontweight='bold')
ax1.legend()
ax1.set_ylim(0, 1)

# Add value labels
for bar, acc in zip(bars, accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{acc:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# --- Plot 2: Sensitivity by Group ---
ax2 = axes[0, 1]
sensitivities = group_metrics['Sensitivity'].tolist()

bars = ax2.bar(groups, sensitivities, color=bar_colors, edgecolor='black', linewidth=2)
ax2.set_ylabel('Sensitivity (TPR)', fontsize=12)
ax2.set_title('DR Detection Sensitivity by Group', fontsize=14, fontweight='bold')
ax2.set_ylim(0, 1)

for bar, sens in zip(bars, sensitivities):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{sens:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# --- Plot 3: QWK by Group ---
ax3 = axes[1, 0]
qwks = group_metrics['QWK'].tolist()

bars = ax3.bar(groups, qwks, color=bar_colors, edgecolor='black', linewidth=2)
ax3.axhline(y=overall_qwk, color='red', linestyle='--', linewidth=2, label=f'Overall ({overall_qwk:.3f})')
ax3.set_ylabel('Quadratic Weighted Kappa', fontsize=12)
ax3.set_title('QWK by Pigmentation Group', fontsize=14, fontweight='bold')
ax3.legend()
ax3.set_ylim(0, 1)

for bar, qwk in zip(bars, qwks):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{qwk:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# --- Plot 4: Fairness Summary ---
ax4 = axes[1, 1]
ax4.axis('off')

fairness_text = f"""
⚖️ FAIRNESS AUDIT SUMMARY
{'='*45}

📊 Demographic Parity (Accuracy)
   • Ratio: {fairness_metrics['demographic_parity_ratio']:.3f}
   • Disparity: {fairness_metrics['accuracy_disparity']:.2%}
   • 80% Rule: {'✓ PASS' if fairness_metrics['passes_80_rule_accuracy'] else '✗ FAIL'}

📊 Equalized Odds (Sensitivity)
   • Ratio: {fairness_metrics['equalized_odds_ratio']:.3f}
   • Disparity: {fairness_metrics['sensitivity_disparity']:.2%}
   • 80% Rule: {'✓ PASS' if fairness_metrics['passes_80_rule_sensitivity'] else '✗ FAIL'}

📊 Additional Metrics
   • QWK Disparity: {fairness_metrics['qwk_disparity']:.3f}
   • Specificity Disparity: {fairness_metrics['specificity_disparity']:.2%}

⚠️ Worst Performing Groups:
   • Accuracy: {fairness_metrics['worst_group_accuracy']}
   • Sensitivity: {fairness_metrics['worst_group_sensitivity']}

{'='*45}
Note: Uses fundus luminance as pigmentation proxy
"""

ax4.text(0.1, 0.5, fairness_text, transform=ax4.transAxes, fontsize=12,
         verticalalignment='center', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Fairness Audit Report: Diabetic Retinopathy Grading', 
             fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'fairness_report.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: {ARTIFACTS_DIR / 'fairness_report.png'}")

## 7. Per-Grade Analysis by Pigmentation

In [ ]:
# Confusion matrix per group
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, group in enumerate(['Light', 'Medium', 'Dark']):
    df_group = df_test[df_test['pigmentation_group'] == group]
    
    if len(df_group) > 0:
        cm = confusion_matrix(df_group['diagnosis'], df_group['prediction'], labels=range(5))
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        
        sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[idx],
                   xticklabels=[f'G{i}' for i in range(5)],
                   yticklabels=[f'G{i}' for i in range(5)])
        axes[idx].set_title(f'{group} Pigmentation (n={len(df_group)})', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Predicted')
        axes[idx].set_ylabel('Actual')

plt.suptitle('Normalized Confusion Matrices by Pigmentation Group', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'fairness_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Saved: {ARTIFACTS_DIR / 'fairness_confusion_matrices.png'}")

## 8. Save Fairness Audit Results

In [ ]:
# Compile audit results
audit_results = {
    'overall_metrics': {
        'accuracy': float(overall_acc),
        'qwk': float(overall_qwk),
        'n_samples': len(df_test)
    },
    'per_group_metrics': group_metrics.to_dict(orient='records'),
    'fairness_metrics': {
        k: bool(v) if isinstance(v, (np.bool_, bool)) else 
           float(v) if isinstance(v, (float, np.floating)) else str(v)
        for k, v in fairness_metrics.items()
    },
    'methodology': {
        'pigmentation_proxy': 'Mean LAB Luminance',
        'stratification': 'Tertile-based (33rd/66th percentile)',
        'fairness_threshold': '80% Rule'
    }
}

# Save to JSON
audit_file = RESULTS_DIR / 'fairness_audit.json'
with open(audit_file, 'w') as f:
    json.dump(audit_results, f, indent=2)

print(f"✅ Saved: {audit_file}")

In [ ]:
# Save fairness module
fairness_code = '''
"""
Fairness Audit Module for Diabetic Retinopathy Grading

Provides tools for:
- Pigmentation proxy estimation
- Per-group performance analysis
- Fairness metrics computation

Author: Deep Retina Grade Project
Date: January 2026
"""

import cv2
import numpy as np
import pandas as pd
from typing import Dict, List, Optional, Tuple
from sklearn.metrics import accuracy_score, cohen_kappa_score


def estimate_pigmentation(img_path: str) -> float:
    """
    Estimate retinal pigmentation from fundus image.
    Uses mean luminance in LAB color space as proxy.
    
    Args:
        img_path: Path to fundus image
        
    Returns:
        Mean luminance value [0, 100]
    """
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    h, w = img.shape[:2]
    margin = int(min(h, w) * 0.1)
    img_cropped = img[margin:h-margin, margin:w-margin]
    
    lab = cv2.cvtColor(img_cropped, cv2.COLOR_RGB2LAB)
    L = lab[:, :, 0]
    mask = L > 10
    
    if mask.sum() == 0:
        return np.nan
    
    return float(L[mask].mean())


def stratify_pigmentation(
    luminance_values: pd.Series,
    bins: int = 3
) -> pd.Series:
    """
    Stratify images by pigmentation level.
    
    Args:
        luminance_values: Series of luminance values
        bins: Number of bins (default: 3)
        
    Returns:
        Series of group labels
    """
    q33 = luminance_values.quantile(0.33)
    q66 = luminance_values.quantile(0.66)
    
    def assign_group(lum):
        if pd.isna(lum):
            return "Unknown"
        elif lum < q33:
            return "Dark"
        elif lum < q66:
            return "Medium"
        else:
            return "Light"
    
    return luminance_values.apply(assign_group)


def compute_group_metrics(
    df: pd.DataFrame,
    group_col: str,
    pred_col: str = "prediction",
    true_col: str = "diagnosis"
) -> pd.DataFrame:
    """
    Compute performance metrics for each group.
    
    Returns:
        DataFrame with per-group metrics
    """
    groups = df[group_col].unique()
    results = []
    
    for group in groups:
        if group == "Unknown":
            continue
            
        mask = df[group_col] == group
        y_true = df.loc[mask, true_col]
        y_pred = df.loc[mask, pred_col]
        
        acc = accuracy_score(y_true, y_pred)
        qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        
        # Binary metrics for DR detection
        y_true_bin = (y_true >= 2).astype(int)
        y_pred_bin = (y_pred >= 2).astype(int)
        
        tp = ((y_true_bin == 1) & (y_pred_bin == 1)).sum()
        fn = ((y_true_bin == 1) & (y_pred_bin == 0)).sum()
        fp = ((y_true_bin == 0) & (y_pred_bin == 1)).sum()
        tn = ((y_true_bin == 0) & (y_pred_bin == 0)).sum()
        
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        results.append({
            "Group": group,
            "N": mask.sum(),
            "Accuracy": acc,
            "QWK": qwk,
            "Sensitivity": sensitivity,
            "Specificity": specificity
        })
    
    return pd.DataFrame(results)


def compute_fairness_metrics(group_metrics: pd.DataFrame) -> Dict:
    """
    Compute fairness metrics from group performance.
    
    Returns:
        Dictionary of fairness metrics
    """
    metrics = {}
    
    # Demographic Parity
    acc_min = group_metrics["Accuracy"].min()
    acc_max = group_metrics["Accuracy"].max()
    metrics["demographic_parity_ratio"] = acc_min / acc_max if acc_max > 0 else 0
    metrics["accuracy_disparity"] = acc_max - acc_min
    
    # Equalized Odds
    sens_min = group_metrics["Sensitivity"].min()
    sens_max = group_metrics["Sensitivity"].max()
    metrics["equalized_odds_ratio"] = sens_min / sens_max if sens_max > 0 else 0
    metrics["sensitivity_disparity"] = sens_max - sens_min
    
    # 80% Rule
    metrics["passes_80_rule_accuracy"] = metrics["demographic_parity_ratio"] >= 0.8
    metrics["passes_80_rule_sensitivity"] = metrics["equalized_odds_ratio"] >= 0.8
    
    return metrics
'''

fairness_file = SRC_DIR / "fairness" / "audit.py"
with open(fairness_file, 'w') as f:
    f.write(fairness_code.strip())

print(f"✅ Saved: {fairness_file}")

## 9. Summary & Recommendations

In [ ]:
print("="*70)
print("⚖️ FAIRNESS AUDIT COMPLETE - SUMMARY")
print("="*70)
print(f"""
✅ Audit Methodology:
   • Proxy: Fundus luminance (LAB L channel)
   • Groups: Light / Medium / Dark (tertile split)
   • Metrics: Accuracy, QWK, Sensitivity, Specificity
   • Fairness: 80% Rule (EEOC standard)

📊 Results Summary:
   • Overall Accuracy: {overall_acc:.2%}
   • Overall QWK: {overall_qwk:.3f}
   • Demographic Parity Ratio: {fairness_metrics['demographic_parity_ratio']:.3f}
   • Equalized Odds Ratio: {fairness_metrics['equalized_odds_ratio']:.3f}

📁 Artifacts Generated:
   • {ARTIFACTS_DIR / 'fairness_report.png'}
   • {ARTIFACTS_DIR / 'pigmentation_distribution.png'}
   • {ARTIFACTS_DIR / 'fairness_confusion_matrices.png'}
   • {RESULTS_DIR / 'fairness_audit.json'}
   • {SRC_DIR / 'fairness' / 'audit.py'}

🔍 Key Findings:
   • Worst group by accuracy: {fairness_metrics['worst_group_accuracy']}
   • Accuracy disparity: {fairness_metrics['accuracy_disparity']:.2%}
   • Sensitivity disparity: {fairness_metrics['sensitivity_disparity']:.2%}
""")

if fairness_metrics['passes_80_rule_accuracy'] and fairness_metrics['passes_80_rule_sensitivity']:
    print("✅ Model PASSES 80% Rule for both accuracy and sensitivity")
else:
    print("⚠️ Model FAILS 80% Rule - bias mitigation recommended")
    print("   Recommendations:")
    print("   1. Data augmentation for underrepresented groups")
    print("   2. Re-weighting training samples")
    print("   3. Adversarial debiasing")
    print("   4. Post-hoc threshold calibration")

print("\n" + "="*70)
print("\n🚀 ALL CORE NOTEBOOKS COMPLETE!")
print("   Next: FastAPI backend + React frontend")
print("="*70)